In [ ]:
import numpy as np
import pandas as pd
import scipy as scp
import numba
import matplotlib as mpl
from matplotlib import pyplot as plt
import os
import json
import trimesh
from scipy import linalg
from scipy.optimize import brentq
from scipy.linalg import svd, inv, hilbert, cholesky, solve_triangular
from scipy.linalg import hilbert
from itertools import product
from scipy.interpolate import RegularGridInterpolator
from scipy.sparse.linalg import cg, LinearOperator
from scipy.optimize import minimize
from scipy import optimize
import logging
from datetime import datetime
import numdifftools as nd
import time
import bisect
# import pylops

In [ ]:
Tianqin_data = pd.read_csv('input/Terrain/SimBox_Nx2_Ny2.csv')
TERRAIN_ACCURACY = 2  # Terrain data resolution in meters

In [ ]:
# user-defined parameters

# numeric options
FLOAT = np.float64
INT = np.int32
REL_GEOM_TOL = 1e-6

# world lattice
X_MIN, X_MAX = (-800., 400.)  # world x range
Y_MIN, Y_MAX = (-500., 400.)  # world y range
Z_MIN, Z_MAX = (-60., 200.)  # world z range
N_X = 120 # number of slices along x
N_Y = 90 # number of slices along y
N_Z = 35 # number of slices along z

# Use terrain elevation data
MEASUREMENT_LOCATION = [[-242.5,133.5,-51.513846],
                        [-197.5,133.5,-51.513846],
                        [-152.5,133.5,-51.513846],
                        [-220,94.5,-51.513846],
                        [-175,94.5,-51.513846]]

MEASUREMENT_DATA = [pd.read_csv('input/SimBox/SimBox_A_2e9_p3_z60_b5_r3_2_13_GaussianFilter.csv'),
                    pd.read_csv('input/SimBox/SimBox_B_2e9_p3_z60_b5_r3_2_16_GaussianFilter.csv'),
                    pd.read_csv('input/SimBox/SimBox_C_2e9_p3_z60_b5_r3_2_21_GaussianFilter.csv'),
                    pd.read_csv('input/SimBox/SimBox_D_2e9_p3_z60_b5_r3_2_21_GaussianFilter.csv'),
                    pd.read_csv('input/SimBox/SimBox_E_2e9_p3_z60_b5_r3_2_21_GaussianFilter.csv')]

# solution initial value and bounds
INIT_DENSITY = 2.6  # density initial guess(g/cm^2)
MIN_DENSITY = 0.  # density lower bound
MAX_DENSITY = 100 # density upper bound in kg/m^3

NOT_OVERLAP_FOLD = False  # Treat all non-overlapping voxels do not having the same density

In [ ]:
# 1. Build parameter dict with automatic NumPy type conversion
def convert_numpy_types(obj):
    if isinstance(obj, np.generic):
        return obj.item()  # Convert scalar (e.g. np.int32 -> int)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj  # Return non-NumPy types as-is
    
# 2. Build parameter dict (collect parameters to save)
params = {
    "X_MIN, X_MAX": (X_MIN, X_MAX),
    "Y_MIN, Y_MAX": (Y_MIN, Y_MAX),
    "Z_MIN, Z_MAX": (Z_MIN, Z_MAX),
    "N_X, N_Y, N_Z": (N_X, N_Y, N_Z),  
    "MEASUREMENT_LOCATION" : MEASUREMENT_LOCATION,
    "INIT_DENSITY, MIN_DENSITY, MAX_DENSITY": (INIT_DENSITY, MIN_DENSITY, MAX_DENSITY),
    "Unit": "(m),(g/cm^2)",
}

# 3. Recursively convert NumPy types in dict
def process_dict(d):
    for key, value in d.items():
        if isinstance(value, dict):
            process_dict(value)
        elif isinstance(value, (list, tuple)):
            # Recursively process each element of list/tuple
            d[key] = [convert_numpy_types(item) for item in value]
        else:
            d[key] = convert_numpy_types(value)
    return d

In [ ]:
class STLModel:
    def __init__(self, filepath: str, density: float, transform_matrix: np.ndarray = None):
        self.filepath = filepath
        self.density = density  # g/cm³
        self.trimesh_obj = trimesh.load_mesh(filepath)
        
        # Apply transformation matrix (position, rotation, scale)
        if transform_matrix is not None:
            self.apply_transform(transform_matrix)
    
    def apply_transform(self, transform_matrix: np.ndarray):
        """Apply 4x4 transformation matrix."""
        self.trimesh_obj.apply_transform(transform_matrix)
        # Also update mesh_data for visualization
        for i, triangle in enumerate(self.trimesh_obj.vectors):
            for j, vertex in enumerate(triangle):
                homogeneous = np.append(vertex, 1)
                transformed = transform_matrix @ homogeneous
                self.trimesh_obj.vectors[i][j] = transformed[:3]

class TerrainReconstructor:
    
    def __init__(self):
        self.stl_models: List[STLModel] = []
    
    def add_stl_model(self, filepath: str, density: float, transform_matrix: np.ndarray = None):
        model = STLModel(filepath, density, transform_matrix)
        self.stl_models.append(model)
        print(f"**AddSTLmodel**: {filepath}, density: {density} g/cm³")


reconstructor = TerrainReconstructor()
# reconstructor.add_stl_model('input/Model/TianqinTunnel_inner_Unit_m.stl', 0)
# reconstructor.add_stl_model('input/Model/TianqinTunnel_Unit_m.stl', 2.3)
# reconstructor.add_stl_model('input/Model/Cube_test.stl', 2.3)

In [ ]:
def setup_logging(verbose_level):
    log_levels = [logging.WARNING, logging.INFO, logging.DEBUG]
    level = log_levels[min(verbose_level, len(log_levels) - 1)]
    
    # Get root logger and set level
    logger = logging.getLogger()
    logger.setLevel(level)
    
    # Add a handler if none exists
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter('%(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    else:
        # Update existing handler levels
        for handler in logger.handlers:
            handler.setLevel(level)

In [ ]:
# parameters and dimension-flatten algorithms

R_MIN: np.ndarray[FLOAT, 3] = np.array([X_MIN, Y_MIN, Z_MIN], dtype=FLOAT)
R_MAX: np.ndarray[FLOAT, 3] = np.array([X_MAX, Y_MAX, Z_MAX], dtype=FLOAT)
N_R: np.ndarray[INT, 3] = np.array([N_X, N_Y, N_Z], dtype=INT)  # number of voxels along each axis
WORLD_DIM: INT = N_X * N_Y * N_Z  # total number of voxels
N_R_MAX: np.ndarray[INT, 3] = N_R - 1
N_R_SUM: INT = N_R[0] * N_R[1] * N_R[2]  # total number of voxels
R_LENGTH: np.ndarray[FLOAT, 3] = R_MAX - R_MIN  # total length along each axis
DU: np.ndarray[FLOAT, 3] = R_LENGTH / N_R  # voxel spacing along each axis
# TEST_OUT = 0

LOCATION: np.ndarray[FLOAT, 3] = np.array(MEASUREMENT_LOCATION, dtype=FLOAT)  # measurement point coordinates
DATASET: np.ndarray = np.array([df.values for df in MEASUREMENT_DATA])  # measurement datasets (rows: detectors, each cell: one data row)
N_MEASUREMENT: INT = len(DATASET)  # number of measurement datasets
LEN_DATA: INT = np.array([len(data) for data in DATASET])  # data count per dataset, as column vector
N_DOF: INT = LEN_DATA.sum()  # total number of data points
assert (N_MEASUREMENT == len(LOCATION))  # Assert dataset count equals measurement point count

ABS_GEOM_TOL: FLOAT = np.linalg.norm(DU) * REL_GEOM_TOL  # abs tolerance: voxel diagonal length * small positive number
# ABS_GEOM_TOL: FLOAT = 1e-10

ZERO_COUNT = np.sum(DATASET[:, :, 3] == 0)

print(f"Valid data count: {N_DOF - ZERO_COUNT}")
print(f"Invalid data count: {ZERO_COUNT}")
print(f"ABS_GEOM_TOL : {ABS_GEOM_TOL}")
print(f"DU : {DU}")

In [ ]:
@numba.njit
def in_world(r: np.ndarray[FLOAT, 3]) -> bool:  # Check if vector r is within world bounds
    return (R_MIN <= r).all() and (r <= R_MAX).all()


@numba.njit
def r_to_ijk(r: np.ndarray[FLOAT, 3]) -> np.ndarray[INT, 3]:  # Map position r to voxel ijk indices
    return np.maximum(0, np.minimum((N_R * (r - R_MIN) / R_LENGTH).astype(INT), N_R_MAX))

@numba.njit
def ijk_to_r(ijk: np.ndarray[INT, 3]) -> np.ndarray[FLOAT, 3]:
    return (ijk * DU + R_MIN + DU / 2).astype(FLOAT)


def Terrain_average_height(matching_indices: list, data) -> FLOAT:
    heights = data.loc[matching_indices, 'height'].values
    average_height = np.mean(heights)
    return average_height

def find_nearest_point_index(points, x, y):
    target_point = np.array([x, y])
    distances = np.linalg.norm(points - target_point, axis=1)
    nearest_index = np.argmin(distances)
    return nearest_index

def seen_map_terrainmod_execute(data, seen_map_ijk: np.ndarray[bool, N_MEASUREMENT], ijk: np.ndarray[INT, 3], du : np.ndarray[FLOAT, 3], terrainmod : str) -> None:
    if terrainmod == "average":
        xyz = ijk_to_r(ijk)
        x, y, z = xyz
        Xlower_bound = x - du[0]/2
        Xupper_bound = x + du[0]/2
        Ylower_bound = y - du[1]/2
        Yupper_bound = y + du[1]/2

        # Filter rows using boolean conditions
        mask = (data['Y'] >= Ylower_bound) & (data['Y'] <= Yupper_bound) & (data['X'] >= Xlower_bound) & (data['X'] <= Xupper_bound)
        matching_indices = data[mask].index.tolist()

        if len(matching_indices) == 0:
            print(f"No matching indices found for ijk: {ijk}, trying nearest terrainmod")
            return seen_map_terrainmod_execute(data, seen_map_ijk, ijk, du, "nearest")
        height = Terrain_average_height(matching_indices, data)
        if (z > height):
            seen_map_ijk[:] = False
    elif terrainmod == "nearest":
        xyz = ijk_to_r(ijk)
        x, y, z = xyz
        points = np.column_stack([data['X'].values, data['Y'].values])
        index = find_nearest_point_index(points, x, y)
        height = data.loc[index, 'height']
        if (z > height):
            seen_map_ijk[:] = False
    else:
        assert(False), f"terrainmod {terrainmod} not recognized"


def seen_map_terrainmod(data, seen_map: np.ndarray[bool, (N_X, N_Y, N_Z, N_MEASUREMENT)]) -> None:
    terrainmod = "average"
    if np.min([DU[0], DU[1]]) < TERRAIN_ACCURACY:
        terrainmod = "nearest"
    for i in numba.prange(N_X):
        for j in numba.prange(N_Y):
            for k in numba.prange(N_Z):  # Iterate over all voxels
                seen_count: INT = seen_map[i, j, k].sum()
                if seen_count > 0:  # If seen by detector
                    ijk = np.array([i, j, k], dtype=INT)
                    seen_map_terrainmod_execute(data, seen_map[i,j,k], ijk, DU, terrainmod)


def seen_map_introduce_model(reconstructor, seen_map: np.ndarray[bool, (N_X, N_Y, N_Z, N_MEASUREMENT)], model_map: np.ndarray[FLOAT, (N_X, N_Y, N_Z)]) -> INT:
    model_dim: INT = 0
    dx, dy, dz = DU / 2
    for n, model in enumerate(reconstructor.stl_models):
        for i in numba.prange(N_X):
            for j in numba.prange(N_Y):
                for k in numba.prange(N_Z):  # Iterate over all voxels
                    seen_count: INT = seen_map[i, j, k].sum()
                    if seen_count > 0:
                        ijk = np.array([i, j, k], dtype=INT)
                        xyz = ijk_to_r(ijk)
                        if (model.trimesh_obj.contains([xyz])): # If voxel center is inside the model
                            # corners = [
                            #     [xyz[0] + dx_i, xyz[1] + dy_j, xyz[2] + dz_k]
                            #     for dx_i in [-dx, dx]
                            #     for dy_j in [-dy, dy] 
                            #     for dz_k in [-dz, dz]
                            # ]
                            # # if all(model.trimesh_obj.contains(corners)): # If all corners are inside the model
                            # if sum(model.trimesh_obj.contains(corners)) >= 5: # If at least n corners are inside the model
                            #     model_map[i, j, k] = model.density  # Assign model density to voxel
                            #     model_dim += 1

                            # Uniformly sample within voxel; judge by volume fraction
                            sample_density = 5  # samples per dimension, 5^3 = 125 points total
                            sample_points = [
                                [xyz[0] + dx_i, xyz[1] + dy_j, xyz[2] + dz_k]
                                for dx_i in np.linspace(-dx, dx, sample_density)
                                for dy_j in np.linspace(-dy, dy, sample_density)
                                for dz_k in np.linspace(-dz, dz, sample_density)
                            ]
                            
                            in_model = model.trimesh_obj.contains(sample_points)
                            volume_ratio = sum(in_model) / len(in_model)
                            
                            if volume_ratio >= 0.5:
                                model_map[i, j, k] = model.density
                                model_dim += 1
    return model_dim


@numba.njit
def delta_length(r: np.ndarray[FLOAT, 3], l_hat: np.ndarray[FLOAT, 3], ijk: np.ndarray[INT, 3]) -> FLOAT:  # Minimum distance from r to next voxel boundary
    u_low: np.ndarray[FLOAT, 3] = R_MIN + ijk * DU  # Current voxel bounds
    u_up: np.ndarray[FLOAT, 3] = u_low + DU
    t_nested = [[(u_low[a] - r[a]) / l_hat[a]  # `range(2)` below: not check z_low!
                if l_hat[a] != 0 else np.inf for a in range(2)],
                [(u_up[a] - r[a]) / l_hat[a]
                if l_hat[a] != 0 else np.inf for a in range(3)]]
    t_flatten: np.ndarray[FLOAT, 5] = np.array([item for sublist in t_nested for item in sublist],  # Flatten nested list elements
                                               dtype=FLOAT) + ABS_GEOM_TOL
    return np.array([t if t > ABS_GEOM_TOL else np.inf for t in t_flatten]).min()


@numba.njit
def build_seen_map_impl(seen_map: np.ndarray[bool, (N_X, N_Y, N_Z, N_MEASUREMENT)], i_meas: INT):  # (Detect seen voxels, mark with true in seen_map)
    for i_entry in numba.prange(LEN_DATA[i_meas]):  # i_entry: data index; i_meas: detector index
        if DATASET[i_meas][i_entry][3] == 0:  # Skip zero-valued data entries
            continue
        r = LOCATION[i_meas].copy()
        l_hat = DATASET[i_meas][i_entry][0:3]  # Direction vector at current position
        while in_world(r):
            ijk = r_to_ijk(r)
            i, j, k = ijk
            seen_map[i, j, k, i_meas] = True
            r += delta_length(r, l_hat, ijk) * l_hat


@numba.njit
def build_seen_map(seen_map: np.ndarray[bool, (N_X, N_Y, N_Z, N_MEASUREMENT)]) -> None:  # Wrapper: build seen map for all detectors
    for i_meas in numba.prange(N_MEASUREMENT):
        build_seen_map_impl(seen_map, i_meas)


@numba.njit
def build_dimension_and_sboom(seen_map: np.ndarray, n_sboom: np.ndarray, sboom_map: np.ndarray) -> tuple:  # Return (total seen voxels, single-detector-only voxels)
    sol_dim: INT = 0
    if NOT_OVERLAP_FOLD:
        for i in numba.prange(N_X):
            for j in numba.prange(N_Y):
                for k in numba.prange(N_Z):  # Iterate over all voxels
                    seen_count: INT = seen_map[i, j, k].sum()  # Count how many detectors see this voxel
                    if seen_count > 1:
                        sol_dim += 1
                    elif seen_count == 1:  # If seen by only one detector: add to n_sboom, record detector index in sboom_map
                        n_sboom += seen_map[i, j, k]  # n_sboom[i]: count of voxels seen only by detector i
                        for i_meas in numba.prange(N_MEASUREMENT):
                            if seen_map[i, j, k, i_meas]:
                                sboom_map[i, j, k] = i_meas  # sboom_map[ijk] != -1: voxel seen only by that detector
    else:
        for i in numba.prange(N_X):
            for j in numba.prange(N_Y):
                for k in numba.prange(N_Z):  # Iterate over all voxels
                    seen_count: INT = seen_map[i, j, k].sum()  # Count how many detectors see this voxel
                    if seen_count >= 1:
                        sol_dim += 1
    sboom_dim = np.count_nonzero(n_sboom)  # Number of detectors with exclusive voxels
    sol_dim += sboom_dim  
    return (sol_dim, sboom_dim)
    

# @numba.njit
def build_r2s_s2r(seen_map: np.ndarray, n_sboom: np.ndarray, sboom_map: np.ndarray, sboom_dim: INT,
                  sboom_head: np.ndarray, r2s: np.ndarray, s2r: np.ndarray, model_in, model_den, model_map) -> tuple:
    sol_head = 0
    model_dim_inuse = 0
    model_dim_overlap = 0
    for i_meas in numba.prange(N_MEASUREMENT):  # sboom_head: renumber detectors that have exclusive voxels (0..inf)
        if n_sboom[i_meas] > 0:
            sboom_head[i_meas] = sol_head
            sol_head += 1
    i_sol = sboom_dim  # Exclusive voxels share same index as their sole detector

    if NOT_OVERLAP_FOLD:
        for i in numba.prange(N_X):
            for j in numba.prange(N_Y):
                for k in numba.prange(N_Z):
                    seen_count: INT = seen_map[i, j, k].sum()
                    density: FLOAT = model_map[i, j, k]  # voxel density
                    if density >= 0:  # If voxel has a density value
                        r2s[i, j, k] = i_sol
                        s2r[i_sol] = [i, j, k]
                        model_in.append(i_sol)
                        model_den.append(density)
                        i_sol += 1
                        model_dim_inuse += 1
                        if seen_count > 1:
                            model_dim_overlap += 1
                    elif seen_count > 1:  # Place multi-detector voxels after exclusive ones; build bidirectional index
                        r2s[i, j, k] = i_sol  # Map voxel ijk to solution vector index
                        s2r[i_sol] = [i, j, k]  # Map solution vector index to voxel ijk
                        i_sol += 1
                    elif seen_count == 1:  # If seen by only one detector
                        r2s[i, j, k] = sboom_head[sboom_map[i, j, k]]  # Use renumbered detector index for exclusive voxels
    else:
        for i in numba.prange(N_X):
            for j in numba.prange(N_Y):
                for k in numba.prange(N_Z):
                    seen_count: INT = seen_map[i, j, k].sum()
                    density: FLOAT = model_map[i, j, k]  # voxel density
                    if density >= 0:  # If voxel has a density value
                        r2s[i, j, k] = i_sol
                        s2r[i_sol] = [i, j, k]
                        model_in.append(i_sol)
                        model_den.append(density)
                        i_sol += 1
                        model_dim_inuse += 1
                        model_dim_overlap += 1
                    elif seen_count >= 1:  
                        r2s[i, j, k] = i_sol  # Map voxel ijk to solution vector index
                        s2r[i_sol] = [i, j, k]  # Map solution vector index to voxel ijk
                        i_sol += 1
    return (i_sol, model_dim_inuse, model_dim_overlap)  # Return actual voxel count and model voxel count
                    




def analyse_geometry() -> tuple:
    seen_map = np.zeros(shape=(N_X, N_Y, N_Z, N_MEASUREMENT), dtype=bool)
    model_map = -np.ones(shape=(N_X, N_Y, N_Z,), dtype=FLOAT)  # model_map: density value per voxel (-1 = no model)
    print("**Begin building seen map...**")
    build_seen_map(seen_map)  # Build visibility map: which voxels are seen and by how many detectors
    seen_map_terrainmod(Tianqin_data, seen_map)  # Voxels above terrain surface treated as unseen
    print("**Seen map built.**")
    print("**Begin introducing model...**")
    model_dim = seen_map_introduce_model(reconstructor, seen_map, model_map)  # Substitute known model densities
    print(f"**Model introduced, model dimension: {model_dim}**")
    # number of lattices be and only be seen by a measurement
    # SBOOM: seen-by-only-one-measurement
    n_sboom = np.zeros(N_MEASUREMENT, dtype=INT)
    # SBOOM lattice index to measurement index
    sboom_map = -np.ones(shape=N_R, dtype=INT)
    sol_dim, sboom_dim = build_dimension_and_sboom(
        seen_map, n_sboom, sboom_map)

    sboom_head = -np.ones(N_MEASUREMENT, dtype=INT)
    s2r = -np.ones(shape=(N_R_SUM, 3), dtype=INT)
    r2s = -np.ones(shape=N_R, dtype=INT)
    model_in = []
    model_den = []
    print("**Begin building r2s and s2r maps...**")
    actual_dim, model_dim_inuse, model_dim_overlap = build_r2s_s2r(seen_map, n_sboom, sboom_map, sboom_dim, 
                  sboom_head, r2s, s2r, model_in, model_den, model_map)
    print("**r2s and s2r maps built.**")
    model_in = np.array(model_in, dtype=INT)
    model_den = np.array(model_den, dtype=FLOAT)
    return (seen_map, n_sboom, sboom_map, sol_dim, sboom_dim, model_dim, model_dim_inuse, model_dim_overlap, actual_dim, sboom_head, r2s, s2r, model_in, model_den)

SEEN_MAP, N_SBOOM, SBOOM_MAP, SOL_DIM, SBOOM_DIM, MODEL_DIM, MODEL_DIM_INUSE, MODEL_DIM_OVERLAP, ACTUAL_DIM, SBOOM_HEAD, REAL_TO_SOL, SOL_TO_REAL, MODEL_INDEX, MODEL_DENSITY = analyse_geometry()
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
print('World dimensions:', WORLD_DIM)
print('Solution dimensions:', SOL_DIM)
print('SBOOM dimensions:', SBOOM_DIM)
print('Model dimensions:', MODEL_DIM)
print('Model dimensions in use:', MODEL_DIM_INUSE)
print('Independent dimensions:', SOL_DIM - SBOOM_DIM - MODEL_DIM_OVERLAP)
print('Actual dimensions:', ACTUAL_DIM)
assert (SOL_DIM == ACTUAL_DIM - MODEL_DIM_INUSE + MODEL_DIM_OVERLAP), "error, dimmension calculate logic is not solid" # Assert consistency of dimension calculation
print('Averaged dimensions:', f'{N_SBOOM.sum()} -> {SBOOM_DIM}')
print('flated index:', f'{SBOOM_HEAD}')
print('Model index:', MODEL_INDEX)
print('Model density:', MODEL_DENSITY)
# print('\n')
# print('World index -> solution index:')
# print(REAL_TO_SOL)
# print('\n')
# print('Solution index -> world index:')
# print(SOL_TO_REAL)

In [ ]:
# build linear system

@numba.njit  # row: stores path length through each voxel at corresponding index
def update_length_matrix_row(row: np.ndarray[FLOAT], r0: np.ndarray[FLOAT, 3], l_hat: np.ndarray[FLOAT, 3]) :#-> FLOAT:
    r: np.ndarray[FLOAT, 3] = r0.copy()  # Create a copy of r0
    while in_world(r):
        ijk = r_to_ijk(r)
        dl = delta_length(r, l_hat, ijk)
        i, j, k = ijk
        if REAL_TO_SOL[i, j, k] != -1:
            row[REAL_TO_SOL[i, j, k]] += dl * 100. # Multiply by 100 to convert m to cm
            # row[REAL_TO_SOL[i, j, k]] += dl
        r += dl * l_hat 


@numba.njit
def build_dense_linear_system(length_mat: np.ndarray[FLOAT, (N_DOF, ACTUAL_DIM)],  # N_DOF: total number of data points
                              sigma_vec: np.ndarray[FLOAT], 
                              sigma_vec_error: np.ndarray[FLOAT]) -> INT:
    ndf_filter = N_DOF
    row_index = 0
    for i in numba.prange(N_MEASUREMENT):
        location = LOCATION[i]
        data = DATASET[i]
        for j in range(LEN_DATA[i]):
            entry = data[j]
            if entry[3] == 0:
                ndf_filter -= 1  # Skip zero-valued entries in DOF count
                continue
            direction = entry[0:3]  # direction: first 3 columnss
            sigma_vec[row_index] = entry[3]  # sigma: the 4th column
            sigma_vec_error[row_index] = entry[4]  # sigma error: the 5th column
            update_length_matrix_row(length_mat[row_index], location, direction)  # Store sigma and path lengths for each detector/direction
            row_index += 1
            # length_mat[i]: path length through each voxel for data entry i along this ray
            # Store sigma in traversal order
    return ndf_filter


@numba.njit  # Extract row, col, data from length_mat for sparse matrix
def build_sparse_length_matrix_index(length_mat: np.ndarray, ndof_new: INT,
                                     row: np.ndarray, col: np.ndarray, data: np.ndarray):
    i_sparse: INT = 0
    for i in numba.prange(ndof_new):
        for j in numba.prange(ACTUAL_DIM):
            if length_mat[i, j] >= ABS_GEOM_TOL:
                row[i_sparse] = i
                col[i_sparse] = j
                data[i_sparse] = length_mat[i, j]
                i_sparse += 1


def build_sparse_linear_system() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:  # Convert length_mat to sparse matrix for efficient storage and computation
    length_mat = np.zeros(shape=(N_DOF, ACTUAL_DIM), dtype=FLOAT)
    sigma_vec = np.zeros(N_DOF, dtype=FLOAT)
    sigma_vec_error = np.zeros(N_DOF, dtype=FLOAT)
    ndof_new = build_dense_linear_system(length_mat, sigma_vec, sigma_vec_error)
    length_mat = length_mat[:ndof_new, :]
    sigma_vec = sigma_vec[:ndof_new]
    sigma_vec_error = sigma_vec_error[:ndof_new]

    n_nonzero = np.count_nonzero(length_mat >= ABS_GEOM_TOL)
    row = -np.ones(n_nonzero, dtype=INT)
    col = -np.ones(n_nonzero, dtype=INT)
    data = np.ndarray(n_nonzero, dtype=FLOAT)
    build_sparse_length_matrix_index(length_mat, ndof_new, row, col, data)

    print(f"ndof_new = {ndof_new}, N_DOF = {N_DOF}, difference = {N_DOF - ndof_new}")
    sparse_length_mat = scp.sparse.coo_matrix(
        (data, (row, col)), shape=(ndof_new, ACTUAL_DIM))

    return (length_mat, sparse_length_mat, sigma_vec, sigma_vec_error, ndof_new)


# _, LENGTH_MATRIX, SIGMA_VECTOR, SIGMA_VECTOR_ERROR, N_DOF_REAL = build_sparse_linear_system()
length_mat, LENGTH_MATRIX, SIGMA_VECTOR, SIGMA_VECTOR_ERROR, N_DOF_REAL = build_sparse_linear_system()
print(LENGTH_MATRIX.toarray().shape)
print(f"LENGTH_MATRIX has inf values: {np.isinf(length_mat).any()}")
# print(f"Matrix rank: {np.linalg.matrix_rank(length_mat)}")

In [ ]:
def is_symmetric(matrix):
    # Compute matrix transpose
    transpose_matrix = np.transpose(matrix)

    # Compare matrix with its transpose
    return np.array_equal(matrix, transpose_matrix)

def compute_inverse_cholesky(C):
    n = len(C)
    L_C = cholesky(C, lower=True)
    I = np.eye(n)
    Y = solve_triangular(L_C, I, lower=True)
    C_inv = solve_triangular(L_C.T, Y, lower=False)
    return C_inv

def plot_SA_history(history):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    axes[0, 0].plot(history['iteration'], history['temperature'])
    axes[0, 0].set_yscale('log')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Temperature')
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['iteration'], history['energy'])
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Energy (chi2)')
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['iteration'], history['acceptance_rate'])
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Cumulative Acceptance Rate')
    axes[1, 0].grid(True)
    
    axes[1, 1].scatter(history['temperature'], history['energy'], alpha=0.5, s=1)
    axes[1, 1].set_xscale('log')
    axes[1, 1].set_xlabel('Temperature')
    axes[1, 1].set_ylabel('Energy')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()


def construct_C_matrix(s2r, sigma_rho, lambda_rho):
    n = s2r.shape[0]
    # Build C matrix
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            iindex_i, iindex_j, iindex_k = s2r[i]
            jindex_i, jindex_j, jindex_k = s2r[j]
            x_i, y_i, z_i = ijk_to_r(np.array([iindex_i, iindex_j, iindex_k], dtype=INT))
            x_j, y_j, z_j = ijk_to_r(np.array([jindex_i, jindex_j, jindex_k], dtype=INT))
            C[i, j] = (sigma_rho ** 2) * np.exp(-np.linalg.norm([x_i - x_j, y_i - y_j, z_i - z_j]) / lambda_rho)
    return C

def construct_C_weighted(C_inv, initial_rho, x):
    weight = C_inv @ (x - initial_rho)
    return weight

def construct_difference_weighted(initial_rho, x):
    weight = (x - initial_rho)
    return weight

def construct_squared_difference_weighted(initial_rho, x):
    weight = (x - initial_rho)**2
    return weight

def construct_parameter_vector(x, sigma_rho, lambda_rho):
    """
    Merge x and C parameters into a single parameter vector
    """
    return np.concatenate([x, [sigma_rho], [lambda_rho]])

def extract_parameters(full_params, n):
    """
    Extract x and C parameters from full parameter vector
    """
    x = full_params[:n]  # First n elements are x
    sigma_rho, lambda_rho = full_params[n:]  # Remaining elements are C parameters
    return x, sigma_rho, lambda_rho

@numba.njit
def generate_gaussian(n, sigma=1.0): # Generate normalized 3D Gaussian convolution kernel (n: radius in voxels)
    core = np.array([np.exp(-(i**2 + j**2 + k**2) / (2 * sigma**2)) for i in range(-n, n+1) for j in range(-n, n+1) for k in range(-n, n+1)])
    core = core / np.sum(core)
    return core

def chisquare_analyze(A, b, x, dof, errors = None, params = {}):
    residuals = b - A @ x
    if errors is not None:
        residuals = residuals / np.abs(errors)
    chi_squared = np.sum(residuals**2)
    if dof < 0:
        logging.warning("Warning: Degrees of freedom < 0, possibly due to too many unknown variables or insufficient data.")
        dof = 0
    reduced_chi_squared = chi_squared / dof if dof > 0 else float('inf')
    if 0.8 <= reduced_chi_squared <= 1.2:
        logging.info("✓ Reduced chi-squared near 1, good fit.")
    elif reduced_chi_squared > 1.2:
        logging.info("⚠ Reduced chi-squared > 1, possible underfitting or underestimated weights.")
    else:
        logging.info("⚠ Reduced chi-squared < 1, possible overfitting or overestimated weights.")
    
    logging.info(f"χ²: {chi_squared:.6f}")
    logging.info(f"Degrees of freedom: {dof}")
    logging.info(f"Reduced chi-squared: {reduced_chi_squared:.6f}")

    if params != {}:
        params["Degrees_of_Freedom"] = dof
        params["Chi_Squared"] = chi_squared
        params["Reduced_Chi_Squared"] = reduced_chi_squared

    return chi_squared, reduced_chi_squared

def find_lambda_rho(AtWA, ATWb, A_unknown, b_modified, method, lambda_range=None, plot=False):
    if lambda_range is None:
        # L-curve method parameters
        n_lambda = 100  # Number of candidate lambda values
        
        # Define lambda search range (log-spaced)
        lambda_min = 1e-15
        lambda_max = 1e5
        lambda_candidates = np.logspace(np.log10(lambda_min), np.log10(lambda_max), n_lambda)
    else:
        lambda_candidates = np.array(lambda_range)
        
    # Store results
    residual_norms = []
    solution_norms = []
    valid_lambdas = []
    skip_count = 0
    
    
    for i, lambda_reg in enumerate(lambda_candidates):
        try:
            # Solve regularized problem
            regularized_matrix = AtWA + lambda_reg * np.eye(A_unknown.shape[1])
            
            # Check numerical stability
            cond_num = np.linalg.cond(regularized_matrix)
            if cond_num > 1e12:
                skip_count += 1
                logging.debug(f"Skipping lambda {lambda_reg:.2e} due to ill-conditioning (condition number: {cond_num:.2e})")
                continue
            
            x_lambda = np.linalg.solve(regularized_matrix, ATWb)
            
            # Compute residual norm and solution norm
            residual = A_unknown @ x_lambda - b_modified
            residual_norm = np.linalg.norm(residual)
            solution_norm = np.linalg.norm(x_lambda)
            
            # Check result validity
            if np.isfinite(residual_norm) and np.isfinite(solution_norm) and solution_norm > 0:
                residual_norms.append(residual_norm)
                solution_norms.append(solution_norm)
                valid_lambdas.append(lambda_reg)
            else:
                logging.warning(f"Invalid norms for lambda {lambda_reg:.2e}: residual_norm={residual_norm}, solution_norm={solution_norm}")
                skip_count += 1
                
        except Exception as e:
            continue
    
    if len(valid_lambdas) < 5:
        logging.warning(f"Warning: Too few valid lambda values for L-curve analysis, choosing initial lambda: {lambda_reg}.")
        lambda_opt = lambda_reg
    else:
        # Convert to numpy array and take log
        residual_norms = np.array(residual_norms)
        solution_norms = np.array(solution_norms)
        valid_lambdas = np.array(valid_lambdas)

        log_residuals = np.log10(residual_norms)
        log_solutions = np.log10(solution_norms)

        
        def compute_curvature(x, y, valid_lambdas):
            """Compute discrete curve curvature."""
            # Compute 1st and 2nd derivatives (central difference)
            dx = np.gradient(x, valid_lambdas)
            dy = np.gradient(y, valid_lambdas)
            ddx = np.gradient(dx, valid_lambdas)
            ddy = np.gradient(dy, valid_lambdas)
            
            # Curvature: kappa = |x'y'' - y'x''| / (x'^2 + y'^2)^(3/2)
            numerator = np.abs(dx * ddy - dy * ddx)
            denominator = (dx**2 + dy**2)**(3/2)
            curvature = numerator / denominator
            
            return curvature
            
        def find_optimal_lambda_curvature(residual_norms, solution_norms, valid_lambdas):
            """Use improved curvature method to find optimal lambda."""
            
            try:
                # Compute curvature
                curvature = compute_curvature(residual_norms, solution_norms, valid_lambdas)
                
                if len(curvature) <= 2:
                    # Too few points: return middle value
                    return valid_lambdas[len(valid_lambdas)//2]
                
                # Search only in interior points
                inner_curvature = curvature[1:-1]
                inner_lambdas = valid_lambdas[1:-1]
                
                # Method 1: find local maxima
                peaks, properties = scp.signal.find_peaks(inner_curvature, 
                                            height=np.max(inner_curvature)*0.1,  # At least 10% of max
                                            distance=1)  # Adjacent peaks separated by at least 1 point
                
                if len(peaks) > 0:
                    # If multiple peaks, choose one with max curvature
                    best_peak_idx = peaks[np.argmax(inner_curvature[peaks])]
                    lambda_opt = inner_lambdas[best_peak_idx]
                else:
                    # If no clear peak, use global max among interior points
                    max_curvature_idx = np.argmax(inner_curvature)
                    lambda_opt = inner_lambdas[max_curvature_idx]
                    
                return lambda_opt, inner_curvature
                
            except Exception as e:
                logging.warning(f"curvatureComputefailed: {e}")
                return valid_lambdas[len(valid_lambdas)//2]
            
        lambda_opt_curvature, curvature = find_optimal_lambda_curvature(log_residuals, log_solutions, valid_lambdas)
        
        # Method 2: corner detection on L-curve
        def find_corner_point(x, y):
            """findLcurvecorner"""
            n = len(x)
            if n < 3:
                return n // 2
                
            # Distance from each point to line connecting endpoints
            x_start, y_start = x[0], y[0]
            x_end, y_end = x[-1], y[-1]
            
            # Line equation coefficients
            A = y_end - y_start
            B = x_start - x_end
            C = x_end * y_start - x_start * y_end
            
            distances = []
            for i in range(n):
                # Point-to-line distance
                dist = abs(A * x[i] + B * y[i] + C) / np.sqrt(A**2 + B**2)
                distances.append(dist)
            
            return np.argmax(distances)
        
        try:
            corner_idx = find_corner_point(residual_norms, solution_norms)
            lambda_opt_corner = valid_lambdas[corner_idx]
        except:
            lambda_opt_corner = valid_lambdas[len(valid_lambdas)//2]

        # Combined optimal lambda selection (weights adjustable)
        candidates = [lambda_opt_curvature, lambda_opt_corner]
        # Use median for robustness
        # lambda_opt = np.mean(candidates)
        lambda_opt = lambda_opt_curvature
        # lambda_opt = lambda_opt_corner

        print(f"L-curve analysis results:")
        print(f"  Curvature method: λ = {lambda_opt_curvature:.6e}")
        print(f"  Corner method: λ = {lambda_opt_corner:.6e}")
        print(f"  Final choice: λ = {lambda_opt:.6e}")
        print(f" Valid lambdas: {len(valid_lambdas)} (skipped {skip_count} due to ill-conditioning)")

        if plot:
            try:
                plt.figure(figsize=(17, 8))
                
                plt.subplot(1, 2, 1)
                plt.loglog(residual_norms, solution_norms, 'b.-', alpha=0.7)
                plt.xlabel('Residual Norm ||Ax - b||')
                plt.ylabel('Solution Norm ||x||')
                plt.title('L-curve')
                plt.grid(True, alpha=0.3)
                
                # Mark selected point
                opt_idx = np.argmin(np.abs(valid_lambdas - lambda_opt_curvature))
                plt.loglog(residual_norms[opt_idx], solution_norms[opt_idx], 'ro', markersize=8, label=f'Curvature λ={lambda_opt_curvature:.2e}')
                plt.legend()

                # opt_idx = np.argmin(np.abs(valid_lambdas - lambda_opt_corner))
                # plt.loglog(residual_norms[opt_idx], solution_norms[opt_idx], 'go', markersize=8, label=f'Corner λ={lambda_opt_corner:.2e}')
                # plt.legend()
                
                plt.subplot(1, 2, 2)
                if len(curvature) > 0:
                    plt.semilogx(valid_lambdas[1:len(valid_lambdas)-1], curvature, 'g.-')
                    plt.axvline(lambda_opt, color='r', linestyle='--', alpha=0.7)
                    plt.xlabel('Regularization Parameter λ')
                    plt.ylabel('Curvature')
                    plt.title('L-curve Curvature')
                    plt.grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.savefig('output/l_curve_analysis.png', dpi=1200)
                plt.show()
                
                
            except ImportError:
                logging.warning("matplotlib not available, skipping L-curve plot")
            except Exception as e:
                logging.warning(f"Error plotting L-curve: {e}")
            
        iter_count = len(valid_lambdas)  # Record number of lambda values evaluated
        logging.debug(f"Optimal lambda found: {lambda_opt:.6e} after {iter_count} iterations")
        return lambda_opt

def solve_linear_raw(A, b, error):
    weights = np.diag(1.0 / (error**2))
    AtWA = A.T @ weights @ A
    ATWb = A.T @ weights @ b
    regularized_matrix = AtWA
    x_solution = np.linalg.solve(AtWA, ATWb)

    if np.linalg.matrix_rank(regularized_matrix) == regularized_matrix.shape[1] == regularized_matrix.shape[0]:
        Cov_x = scp.linalg.inv(regularized_matrix)
        # Cov_x = scp.linalg.pinv(regularized_matrix)
    else:
        logging.info("Warning: Matrix is not full rank; covariance matrix may be unstable.")
        Cov_x = np.zeros((A.shape[1], A.shape[1]))
    
    # Parameter standard deviation
    x_errors = np.sqrt(np.diag(Cov_x))
    return x_solution, x_errors

def solve_linear_C(A, b, error, sol_to_real, sigma_rho, lambda_rho, x0, params = {}):
    weights = np.diag(1.0 / (error**2))
    AtWA = A.T @ weights @ A
    ATWb = A.T @ weights @ b
    C = construct_C_matrix(sol_to_real, sigma_rho, lambda_rho)
    if not is_symmetric(C):
        logging.debug(f"C is symmetric: {is_symmetric(C)}")
    assert(np.linalg.matrix_rank(C) == C.shape[0]), f"C rank deficiency, C rank = {np.linalg.matrix_rank(C)}, sigma_rho = {sigma_rho}, lambda_rho = {lambda_rho}"
    ATWbC = ATWb + scp.linalg.solve(C, x0, assume_a='pos')
    Cinv = compute_inverse_cholesky(C)
    regularized_matrix = AtWA + Cinv
    # regularized_matrix = AtWA + C
    x_solution = np.linalg.solve(regularized_matrix, ATWbC)

    if np.linalg.matrix_rank(regularized_matrix) == regularized_matrix.shape[1] == regularized_matrix.shape[0]:
        Cov_x = scp.linalg.inv(regularized_matrix)
        # Cov_x = scp.linalg.pinv(regularized_matrix)
    else:
        logging.info("Warning: Matrix is not full rank; covariance matrix may be unstable.")
        Cov_x = np.zeros((A.shape[1], A.shape[1]))
    
    # Parameter standard deviation
    x_errors = np.sqrt(np.diag(Cov_x))
    # x_errors = np.zeros(shape=x_solution.shape)

    if params != {}:
        new_params = {}
        new_params["Sigma_Rho"] = sigma_rho
        new_params["Lambda_Rho"] = lambda_rho
        new_params["initial_rho"] = x0[0]
        params.update(new_params)
    return x_solution, x_errors

def solve_linear_nelder(A, b, error, x0, n, sol_to_real, bounds, sigma_rho, lambda_rho, rho_0=None):
    weights = np.diag(1.0 / (error**2))

    if rho_0 is None:
        if bounds is not None:
            assert(len(bounds)-2 == A.shape[1]), "bounds length should be equal to number of matrix columns + 2"
        def chisquare(full_params, A, b, W, x_0, n, sol_to_real):
                x, sigma_rho, lambda_rho = extract_parameters(full_params, n)
                C = construct_C_matrix(sol_to_real, sigma_rho, lambda_rho)
                if not is_symmetric(C):
                    logging.debug(f"C is symmetric: {is_symmetric(C)}")
                # assert(np.linalg.matrix_rank(C) == C.shape[0]), f"C rank deficiency, C rank = {np.linalg.matrix_rank(C)}, sigma_rho = {sigma_rho}, lambda_rho = {lambda_rho}"
                residual = A @ x - b
                chi2_data = residual.T @ W @ residual
                diff = x - x_0
                y = scp.linalg.solve(C, diff, assume_a='pos')
                chi2_prior = diff.T @ y
                chi2_total = chi2_data + chi2_prior
                return chi2_total
        initial_params = construct_parameter_vector(x0, sigma_rho, lambda_rho)
    else:
        if bounds is not None:
            assert(len(bounds) == 2), "bounds length should be equal to 2"
        def chisquare(full_params, rho_0, A, b, W, x_0, n, sol_to_real):
                sigma_rho, lambda_rho = full_params[0], full_params[1]
                C = construct_C_matrix(sol_to_real, sigma_rho, lambda_rho)
                if not is_symmetric(C):
                    logging.debug(f"C is symmetric: {is_symmetric(C)}")
                # assert(np.linalg.matrix_rank(C) == C.shape[0]), f"C rank deficiency, C rank = {np.linalg.matrix_rank(C)}, sigma_rho = {sigma_rho}, lambda_rho = {lambda_rho}"
                residual = A @ rho_0 - b
                chi2_data = residual.T @ W @ residual
                diff = rho_0 - x_0
                y = scp.linalg.solve(C, diff, assume_a='pos')
                chi2_prior = diff.T @ y
                chi2_total = chi2_data + chi2_prior
                return chi2_total
        initial_params = np.concatenate([[sigma_rho], [lambda_rho]])

    
    def callback(intermediate_result):
        global previous_chi_square, iteration_count

        current_chi_square = intermediate_result.fun
        chi_change = None
        if previous_chi_square is not None:
            chi_change = abs(current_chi_square - previous_chi_square)
        
        iteration_count += 1
        
        print(f"Iteration {iteration_count}:")
        print(f"  Current chi-square = {current_chi_square:.6e}")
        if chi_change is not None:
            print(f"  Change in chi-square = {chi_change:.6e}")
        
        # Update previous chi-squared value
        previous_chi_square = current_chi_square

    

    result = minimize(
        lambda params: chisquare(params, rho_0, A, b, weights, x0, n, sol_to_real) if rho_0 is not None else chisquare(params, A, b, weights, x0, n, sol_to_real),
        initial_params,
        method='nelder-mead',
        callback=callback,
        bounds=bounds,
        options={
            'maxiter': 1000000,
            'xatol': 1e-4,
            'fatol': 1e-4,
        }
    )

    if not result.success:
        logging.error("Optimization failed: " + result.message)
        raise RuntimeError("Optimization did not converge")
    print(f"Optimization successful: {result.message}")

    if rho_0 is None:
        x_solution, sigma_rho_opt, lambda_rho_opt = extract_parameters(result.x, n)
        return x_solution, sigma_rho_opt, lambda_rho_opt, result
    else:
        sigma_rho_opt, lambda_rho_opt = result.x[0], result.x[1]
        return sigma_rho_opt, lambda_rho_opt

def chisquare_for_brute(x, rho_0, A, b, weights, x0, sol_to_real):
    sigma_rho, lambda_rho = x
    C = construct_C_matrix(sol_to_real, sigma_rho, lambda_rho)
    if not is_symmetric(C):
        logging.debug(f"C is symmetric: {is_symmetric(C)}")
    # assert(np.linalg.matrix_rank(C) == C.shape[0]), f"C rank deficiency, C rank = {np.linalg.matrix_rank(C)}, sigma_rho = {sigma_rho}, lambda_rho = {lambda_rho}"
    residual = A @ rho_0 - b
    chi2_data = residual.T @ weights @ residual
    diff = rho_0 - x0
    y = scp.linalg.solve(C, diff, assume_a='pos')
    chi2_prior = diff.T @ y
    chi2_total = chi2_data + chi2_prior
    return chi2_total

def solve_linear_brute(A, b, error, x0, sol_to_real, bounds, rho_0, Ns=20):
    weights = np.diag(1.0 / (error**2))
    params = (rho_0, A, b, weights, x0, sol_to_real)

    ranges = [
        slice(bounds[0][0], bounds[0][1], complex(0, Ns)),
        slice(bounds[1][0], bounds[1][1], complex(0, Ns))
    ]

    result = optimize.brute(
        chisquare_for_brute,
        args=params,
        ranges=ranges,
        full_output=True,
        Ns=Ns,
        disp=True,
        workers=-1,
        finish=None
    )

    xopt, fval, grid, Jout = result
    sigma_rho_opt, lambda_rho_opt = xopt
    
    logging.info(f"Optimization completed successfully")
    logging.info(f"Optimal parameters: sigma_rho={sigma_rho_opt:.6f}, "
                f"lambda_rho={lambda_rho_opt:.6f}")
    logging.info(f"Minimum chi2: {fval:.6f}")

    return sigma_rho_opt, lambda_rho_opt
    
def solve_linear_L(A, b, error, Lambda, params={}):
    weights = np.diag(1.0 / (error**2))
    AtWA = A.T @ weights @ A
    ATWb = A.T @ weights @ b
    regularized_matrix = AtWA + Lambda * np.eye(A.shape[1])
    x_solution = np.linalg.solve(AtWA, ATWb)

    if np.linalg.matrix_rank(regularized_matrix) == regularized_matrix.shape[1] == regularized_matrix.shape[0]:
        Cov_x = scp.linalg.inv(regularized_matrix)
        # Cov_x = scp.linalg.pinv(regularized_matrix)
    else:
        logging.info("Warning: Matrix is not full rank; covariance matrix may be unstable.")
        Cov_x = np.zeros((A.shape[1], A.shape[1]))
    
    # Parameter standard deviation
    x_errors = np.sqrt(np.diag(Cov_x))

    if params != {}:
        new_params = {}
        new_params["Lambda"] = Lambda
        new_params["initial_rho"] = x0[0]
        params.update(new_params)
    return x_solution, x_errors

def solve_sart(A, b, error, x0, relax, bounds = None, niter = 1e3, tof = 1e-6, params={}):
    niter = int(niter)
    x = x0.copy()
    A_dense = A.toarray() if hasattr(A, 'toarray') else A
    errors = np.zeros(x.shape)
    for j in range(niter):
        ART = np.zeros(x.shape)# Accumulate ART correction
        w = 0;# Accumulate wi sum  
        Ax = np.dot(A_dense, x)
        residual_vec = b - Ax
        print(f"Iteration: {j}, residual: {np.linalg.norm(residual_vec):.6e}, "
            f"relative: {np.linalg.norm(residual_vec) / np.linalg.norm(b):.6e}")
        if np.linalg.norm(residual_vec) / np.linalg.norm(b) < tof:
            print(f"Converged at iteration {j}")
            break
        for i in range(A_dense.shape[0]):
            # ai = A[i, :]
            ai = A_dense[i, :]
            sigmai = error[i]
            ai_norm = np.linalg.norm(ai)
            if ai_norm > 0:
                residual_i = residual_vec[i]
                wi = np.abs(residual_i/sigmai)
                ART += relax * (residual_i) / ai_norm**2 * ai * wi
                w += wi
        if w > 0:
            x += ART / w
        if bounds is not None:
            x = np.where(x < bounds[0], bounds[0], x)  # Clamp x to min value
            x = np.where(x > bounds[1], bounds[1], x)  # Clamp x to max value
        if j == niter - 1:
            errors = np.abs(ART / w)
    
    if params != {}:
        new_params = {}
        new_params["Niter"] = niter
        new_params["initial_rho"] = x0[0]
        params.update(new_params)
    return x, errors

def solve_sart_optimize(A, b, error, x0, relax, niter = 1e3, tof = 1e-6, params={}):
    niter = int(niter)
    x = x0.copy()
    A_dense = A.toarray() if hasattr(A, 'toarray') else A
    errors = np.zeros(x.shape)
    for j in range(niter):
        ART = np.zeros(x.shape)# Accumulate ART correction
        w = 0;# Accumulate wi sum  
        Ax = np.dot(A_dense, x)
        residual_vec = b - Ax
        print(f"Iteration: {j}, residual: {np.linalg.norm(residual_vec):.6e}, "
            f"relative: {np.linalg.norm(residual_vec) / np.linalg.norm(b):.6e}")
        if np.linalg.norm(residual_vec) / np.linalg.norm(b) < tof:
            print(f"Converged at iteration {j}")
            break
        for i in range(A_dense.shape[0]):
            # ai = A[i, :]
            ai = A_dense[i, :]
            sigmai = error[i]
            ai_norm = np.linalg.norm(ai)
            if ai_norm > 0:
                residual_i = residual_vec[i]
                wi = np.abs(residual_i/sigmai)
                ART += relax * (residual_i) / ai_norm**2 * ai * wi
                w += wi
        if w > 0:
            x += ART / w
        x = np.where(x < 0, np.abs(x), x)  # Clamp x to min value
        if j == niter - 1:
            errors = np.abs(ART / w)
    
    if params != {}:
        new_params = {}
        new_params["Niter"] = niter
        new_params["initial_rho"] = x0[0]
        params.update(new_params)
    return x, errors

def solve_scp_minimize(A, b, error, x0, methods, bounds = None, params = {}, niter = 1e3):
    weights = np.diag(1.0 / (error**2))

    def objective(x):
        residual = A @ x - b
        chi2 = residual.T @ weights @ residual
        return chi2

    def callback(intermediate_result):
        global previous_chi_square, iteration_count

        current_chi_square = intermediate_result.fun
        chi_change = None
        if previous_chi_square is not None:
            chi_change = abs(current_chi_square - previous_chi_square)
        
        iteration_count += 1
        
        print(f"Iteration {iteration_count}:")
        print(f"  Current chi-square = {current_chi_square:.6e}")
        if chi_change is not None:
            print(f"  Change in chi-square = {chi_change:.6e}")
        
        # Update previous chi-squared value
        previous_chi_square = current_chi_square
    
    result = minimize(
        objective,
        x0,
        method=methods,
        bounds=bounds,
        callback=callback,
        options={
            'maxiter': niter,
            # 'xtol': 1e-6,
            # 'gtol': 1e-6
        }
    )

    # errors = get_reliable_errors(objective, result.x)
    errors = np.zeros(x0.shape)

    if params != {}:
        new_params = {}
        new_params["Niter"] = niter
        new_params["initial_rho"] = x0[0]
        params.update(new_params)
    return result.x, errors

def get_reliable_errors(chi2_func, x_solution):
    hessian_func = nd.Hessian(chi2_func)
    H = hessian_func(x_solution)
    
    # For chi-squared fit, covariance = inverse of half Hessian
    try:
        cov_matrix = np.linalg.inv(H)
        x_errors = np.sqrt(np.diag(cov_matrix))
        
        # Check positive definiteness
        eigenvalues = np.linalg.eigvals(cov_matrix)
        if np.any(eigenvalues < 0):
            print("Warning: Covariance matrix is not positive definite.")
            
        return x_errors
        
    except np.linalg.LinAlgError:
        print("Warning: Hessian matrix is not invertible, using pseudo-inverse.")
        cov_matrix = np.linalg.pinv(H)
        x_errors = np.sqrt(np.diag(cov_matrix))
        return x_errors

def solve_MLEM(A, b, error, x, bounds = None, niter = 1e3, params={}):
    m, n = A.shape
    A_sum = A.sum(axis=0) + 1e-10

    def objective(x):
        residual = A @ x - b
        chi2 = residual.T @ residual
        return chi2

    def callback(current_chi_square):
        global previous_chi_square, iteration_count

        chi_change = None
        if previous_chi_square is not None:
            chi_change = abs(current_chi_square - previous_chi_square)
        
        iteration_count += 1
        
        print(f"Iteration {iteration_count}:")
        print(f"  Current chi-square = {current_chi_square:.6e}")
        if chi_change is not None:
            print(f"  Change in chi-square = {chi_change:.6e}")
        
        # Update previous chi-squared value
        previous_chi_square = current_chi_square

    for k in range(niter):
        x_old = x.copy()
        correction = 0
        for j in range(m):
            correction += A[j, :] * (b[j] / (np.dot(A[j, :], x_old) + 1e-10))
        x = x_old * correction / A_sum
        callback(objective(x))
        
    
    return x, np.zeros(x.shape)

def solve_MH(A_unknown, b_modified, error, x0, solution_num, T, single_step_size, select_rate, solution_length, fixed_indices, initial_rho, bounds = None, MH_bounds = None, params=params, solution_cutoff = 0, accepted_scale = 1.0, s2r = [], r2s = [], bias_rate = 1., seem_map = None, Cini = None):
    
    @numba.njit
    def Likeihood(x):
        residual = A_unknown @ x - b_modified
        chi2 = residual.T @ np.diag(1.0 / (error**2)) @ residual
        if Cini is not None:
            diff = x - initial_rho
            chi2_prior = diff.T @ Cini @ diff
            chi2 += chi2_prior
        return np.exp(-0.5/T * chi2)
    
    def select_modification_mask(current_x, select_num, seem_p, inv_bound):
        n = len(current_x)
        
        if MH_bounds is not None:
            weights = np.abs(MH_bounds(current_x))
            weights = np.maximum(weights, 1e-10)  # Avoid zero weights
            weights /= weights.sum()
            if seem_p is not None:
                weights *= seem_p
                weights /= weights.sum()
            indices = np.random.choice(n, size=int(select_num), replace=False, p=weights)
        else:
            indices = np.random.choice(n, size=int(select_num), replace=False)
        
        mask = np.zeros(n, dtype=bool)
        mask[indices] = True
        return mask
    
    def generate_proposal_with_gaussian(current_x, mask, core, bias_rate):
        assert len(s2r) == len(current_x), "s2r and r2s mappings must be provided for Gaussian proposal generation"
        assert r2s.shape[0] > 0, "r2s mapping must be provided for Gaussian proposal generation"
        proposal_x = current_x.copy()
    
        for i in range(len(current_x)):
            if not mask[i]:
                continue
                
            x, y, z = s2r[i]
            rho_cube = []
            maskin = []
            
            neighborhood = [
                (xi, yi, zi) 
                for xi in range(x-1, x+2)
                for yi in range(y-1, y+2)
                for zi in range(z-1, z+2)
            ]

            offset = 0
            for coord in neighborhood:
                ii, jj, zz = coord
                try:
                    idx = r2s[ii, jj, zz]
                except IndexError:
                    print(f"IndexError for coord: {coord}")
                    idx = -1
                if idx in fixed_indices and len(fixed_indices) > 0:
                    rho_cube.append(0.0)
                    maskin.append(False)
                    continue
                if idx != -1:
                    if len(fixed_indices) > 0:
                        if idx < np.min(fixed_indices):
                            offset = 0
                        else:
                            offset = bisect.bisect_left(fixed_indices, idx)
                    else:
                        offset = 0
                    rho_cube.append(current_x[(idx - offset)])
                    maskin.append(True)
                else:
                    rho_cube.append(0.0)
                    maskin.append(False)
            assert len(rho_cube) == 27
            assert len(maskin) == 27
            rho_cube = np.array(rho_cube)
            maskin = np.array(maskin)
            
            # Normalize kernel and compute weighted sum
            fix_core = core / np.sum(core[maskin])
            lambda_rho = rho_cube @ fix_core
            diff = proposal_x[i] - lambda_rho
            local_step = np.abs(diff)
            
            # if diff > 0:
            #     # Add perturbation
            #     perturbation = np.random.uniform(-local_step, local_step*bias_rate)
            # else:
            #     perturbation = np.random.uniform(-local_step*bias_rate, local_step)

            if diff > 0:
                # Add perturbation
                perturbation = np.random.uniform(-local_step*bias_rate, local_step*bias_rate)
            else:
                perturbation = np.random.uniform(-local_step*bias_rate, local_step*bias_rate)
            
            proposal_x[i] += perturbation
    
        return proposal_x


    @numba.njit
    def rmse(x1, x2):
        return np.sqrt(np.mean((x1 - x2)**2))
    
    def seem_x(seem_map, current_x, s2r):
        seem_p = np.zeros_like(current_x)
        for i in range(len(current_x)):
            x, y, z = s2r[i]
            seem_count = np.sum(seem_map[x, y, z])
            if seem_count > 1:
                seem_p[i] = 1
        return seem_p, np.sum(seem_p)


    valid_solutions = []
    current_exploration = []
    current_x = x0.copy()
    current_likelihood = Likeihood(current_x)
    accepted_count = 0
    recorded_count = 0
    inv_bound = True

    # select_num = int(select_rate * (len(x0)))
    seem_p, seem_count = seem_x(seem_map, current_x, s2r) if seem_map is not None else (None, len(x0))
    print(f"Original solution length: {len(x0)}, SEEM-constrained solution count: {seem_count}")
    select_num = int(select_rate * seem_count)
    print(f"Number of model parameters modified per iteration: {select_num}")
    
    noise_average = error.mean()
    iteration_count = 0
    latest_acceptance_rate = 1
    Total_solutions_needed = solution_num + solution_cutoff
    gaussian_core = generate_gaussian(1, 1.0)


    while len(valid_solutions) + len(current_exploration) < Total_solutions_needed:
        if latest_acceptance_rate < 0.01:
                print(f"⚠️ Local stagnation detected, restarting exploration.")
                if len(current_exploration) > 0: # Pop only if at least one element exists
                    current_exploration.pop()
                    print(f"  Popped last solution. Solutions left: {len(current_exploration)}")
                else:
                    print(f"  Solution collection was empty, nothing to pop.")
                
                current_x = x0.copy()
                current_likelihood = Likeihood(current_x)
                recorded_count = 0
                accepted_count = 0
                latest_acceptance_rate = 1
                valid_solutions.extend(current_exploration[solution_cutoff:])
                current_exploration = []

        if iteration_count > solution_num * solution_length * 100:  # Set upper limit
            print(f"⚠️ Maximum iteration count reached, constraints may be too strict.")
            break
        
        b_now = A_unknown @ current_x
        Rmse = rmse(b_modified, b_now)
        iteration_count += 1
        if iteration_count % 1000 == 0:
            latest_acceptance_rate = accepted_count / 1000
            print(f"M-H: {len(valid_solutions) + len(current_exploration)}/{Total_solutions_needed} | "
                f"Iter: {iteration_count} | Accept: {latest_acceptance_rate:.2%} | "
                f"Current rmse: {Rmse:.2e}")
            accepted_count = 0
        mask = select_modification_mask(current_x, select_num, seem_p, inv_bound)
        proposal_x = generate_proposal_with_gaussian(current_x, mask, gaussian_core, bias_rate)
        if bounds is not None:
            # proposal_x = np.where(proposal_x < bounds[0], bounds[0], proposal_x)  # Clamp x to min value
            proposal_x = np.where(proposal_x < 0, np.abs(proposal_x), proposal_x)  # Clamp x to min value
            # proposal_x = np.where(proposal_x > bounds[1], bounds[1], proposal_x)  # Clamp x to max value
        proposal_likelihood = Likeihood(proposal_x)
        
        acceptance_ratio = proposal_likelihood / (current_likelihood + 1e-20)
        # if acceptance_ratio >= 1:
            # print(f"Proposal likelihood >= current likelihood, notice!")
        # print(f"Proposal likelihood: {proposal_likelihood:.6e}, Current likelihood: {current_likelihood:.6e}")
        # print(f"Acceptance ratio: {acceptance_ratio:.6e}")
        # print(f"rmse: {Rmse:.2e}, noise_average: {noise_average:.6e}")
        if acceptance_ratio >= 1 or np.random.rand() <= acceptance_ratio * accepted_scale:
            # if Rmse <= noise_average * 2.0:
            #     current_x = proposal_x
            #     current_likelihood = proposal_likelihood
            #     recorded_count += 1
            #     accepted_count += 1
            #     print(f"Accepted new proposal. Recorded count: {recorded_count}/{solution_length}")
            # else:
            #     continue
            current_x = proposal_x
            current_likelihood = proposal_likelihood
            recorded_count += 1
            accepted_count += 1
            # print(f"Accepted new proposal. Recorded count: {recorded_count}/{solution_length}")
        else:
            continue 
        
        if recorded_count == solution_length:
            recorded_count = 0
            current_exploration.append(current_x.copy())
            inv_bound = not inv_bound  # Reverse constraint direction
    
    valid_solutions.extend(current_exploration[solution_cutoff:]) 
    valid_solutions = np.array(valid_solutions)
    solution_mean = np.mean(valid_solutions, axis=0)
    solution_error = np.sqrt(np.var(valid_solutions, axis=0))

    if params != {}:
        new_params = {}
        new_params["Solution_Num"] = solution_num
        new_params["Temperature"] = T
        new_params["Single_Step_Size"] = single_step_size
        new_params["Select_Rate"] = select_rate
        new_params["Solution_Length"] = solution_length
        new_params["Solution_Cutoff"] = solution_cutoff
        new_params["Accepted_Scale"] = accepted_scale
        new_params["Bias Rate"] = bias_rate
        params.update(new_params)

    return solution_mean, solution_error

def solve_SA(A_unknown, b_modified, error, x0, T, T_attrate, T_end, single_step_size, select_rate, fixed_indices, bounds = None, MH_bounds = None, bias_rate = 1., params=params, s2r = [], r2s = []):
    

    @numba.njit
    def Likelihood(x):
        residual = A_unknown @ x - b_modified
        chi2 = residual.T @ np.diag(1.0 / (error**2)) @ residual
        return chi2
    
    def select_modification_mask(current_x, select_num):
        n = len(current_x)
        
        if MH_bounds is not None:
            weights = np.abs(MH_bounds(current_x))
            weights = np.maximum(weights, 1e-10)  # Avoid zero weights
            weights /= weights.sum()
            indices = np.random.choice(n, size=int(select_num), replace=False, p=weights)
        else:
            indices = np.random.choice(n, size=int(select_num), replace=False)
        
        mask = np.zeros(n, dtype=bool)
        mask[indices] = True
        return mask

    def generate_proposal_with_gaussian(current_x, mask, core, bias_rate):
        assert len(s2r) == len(current_x), "s2r and r2s mappings must be provided for Gaussian proposal generation"
        assert r2s.shape[0] > 0, "r2s mapping must be provided for Gaussian proposal generation"
        proposal_x = current_x.copy()

        for i in range(len(current_x)):
            if not mask[i]:
                continue
                
            x, y, z = s2r[i]
            rho_cube = []
            maskin = []
            
            neighborhood = [
                (xi, yi, zi) 
                for xi in range(x-1, x+2)
                for yi in range(y-1, y+2)
                for zi in range(z-1, z+2)
            ]
            
            offset = 0
            for coord in neighborhood:
                ii, jj, zz = coord
                try:
                    idx = r2s[ii, jj, zz]
                except IndexError:
                    print(f"IndexError for coord: {coord}")
                    idx = -1
                if idx in fixed_indices and len(fixed_indices) > 0:
                    rho_cube.append(0.0)
                    maskin.append(False)
                    continue
                if idx != -1:
                    if len(fixed_indices) > 0:
                        if idx < np.min(fixed_indices):
                            offset = 0
                        else:
                            offset = bisect.bisect_left(fixed_indices, idx)
                    else:
                        offset = 0
                    rho_cube.append(current_x[idx - offset])
                    maskin.append(True)
                else:
                    rho_cube.append(0.0)
                    maskin.append(False)
            assert len(rho_cube) == 27
            assert len(maskin) == 27
            rho_cube = np.array(rho_cube)
            maskin = np.array(maskin)
            
            # Normalize kernel and compute weighted sum
            fix_core = core / np.sum(core[maskin])
            lambda_rho = rho_cube @ fix_core
            local_step = np.abs(proposal_x[i] - lambda_rho)
            
            # Add perturbation
            perturbation = np.random.uniform(-local_step * bias_rate, local_step * bias_rate)
            proposal_x[i] += perturbation
    
        return proposal_x

    @numba.njit
    def rmse(x1, x2):
        return np.sqrt(np.mean((x1 - x2)**2))
    
    # @numba.njit
    # def MetropolisRate(xnew, xold):
    #     r = np.exp(-((xnew - xold)/T))
    #     # print(f"MetropolisRate: {r}")
    #     return r
    
    history = {
    'iteration': [],
    'temperature': [],
    'energy': [],  # chi-squared value
    'likelihood': [],
    'acceptance_rate': []
    }

    current_x = x0.copy()
    # current_likelihood = Likeihood(current_x)
    accepted_count = 0
    select_num = int(select_rate * (len(x0)))
    print(f"Number of model parameters modified per iteration: {select_num}")
    noise_average = error.mean()
    iteration_count = 0
    latest_acceptance_rate = 1
    gaussian_core = generate_gaussian(1, 1.0)

    while T > T_end and latest_acceptance_rate > 0.01:
        iteration_count += 1
        if iteration_count % 1000 == 0:
            Rmse = rmse(b_modified, A_unknown @ current_x)
            latest_acceptance_rate = accepted_count / 1000
            print(f"current T: {T:.2e} | "
                f"Iter: {iteration_count} | Accept: {latest_acceptance_rate:.2%} | "
                f"Current rmse: {Rmse:.2e}")
            accepted_count = 0
        # start =  time.time()
        mask = select_modification_mask(current_x, select_num)
        # end = time.time()
        # print(f"Mask selection time: {end - start:.6f} seconds")
        proposal_x = generate_proposal_with_gaussian(current_x, mask, gaussian_core, bias_rate)
        if bounds is not None:
            proposal_x = np.where(proposal_x < 0, np.abs(proposal_x), proposal_x)
        proposal_energy = Likelihood(proposal_x)
        current_energy = Likelihood(current_x)
        delta_E = proposal_energy - current_energy
        acceptance_prob = np.exp(-delta_E / T) if delta_E > 0 else 1.0
        # if acceptance_ratio >= 1 or np.random.rand() <= MetropolisRate(proposal_likelihood, current_likelihood):
        if np.random.rand() <= acceptance_prob:
            current_x = proposal_x
            # current_likelihood = proposal_likelihood
            current_energy = proposal_energy
            accepted_count += 1
            # print(f"Accepted new proposal. Recorded count: {recorded_count}/{solution_length}")

        if iteration_count % 10 == 0:  # Log every 10 iterations to limit data volume
            history['iteration'].append(iteration_count)
            history['temperature'].append(T)
            history['energy'].append(current_energy)
            history['acceptance_rate'].append(accepted_count / max(iteration_count, 1))
        T *= (1 - T_attrate)
        bias_rate *= (1 - T_attrate)


    if params != {}:
        new_params = {}
        new_params["Temperature"] = T
        new_params["Single_Step_Size"] = single_step_size
        new_params["Select_Rate"] = select_rate
        new_params["Bias Rate"] = bias_rate
        params.update(new_params)

    return current_x, np.zeros(x0.shape), history

            
    

def solve_linear_L2_straint(A, b, error, sigma_rho, lambda_rho, initial_rho, bounds = None, s2r = [], r2s = [], fixed_indices = [], fixed_values = [], lambda_reg = 0.5, method = 'l_curve', lambda_range = None, plot = True, verbose = 2, params = {}, niter = 1e3, solution_num = 1e5, T = 1., MH_bounds = None, single_step_size = 0.1, select_rate = 0.2, solution_length = 10, solution_cutoff = 5, accepted_scale = 1.0, T_end = 10, T_attrate = 1e-4, bias_rate = 1.0, seem_map = None, x0 = None):
    """
    Solve Ax = b, where some components of x are known.
    Parameters:
    A: design matrix
    b: RHS vector
    error: measurement system error
    lambda_rho: C-constraint correlation length, larger = weaker constraint.
    initial_rho: Estimated density value.
    fixed_indices: Column index list of A matrix for known densities.
    fixed_values: Known density values.
    method: Solvemethod
        # - raw: No regularization or C constraint
        # - C: C constraint only
        # - l_curve: L-curve method for regularization parameter
        # - nelder-mead: Nelder-Mead simplex method
        - profile-maximum-likelihood: First estimate initial rho via raw method, then fix rho and optimize C parameters using Nelder-Mead or brute force.
        - Sart: UseSARTmethoditerationSolve
        - L-BFGS-B: UseL-BFGS-BmethodSolve
        - trust-constr: Usetrust-constrmethodSolve
        - ML-EM: Solve using maximum likelihood expectation maximization.
        - M-H: UseMetropolis-Hastingsmethod（MCMC）Solve
    bounds: Variable x boundary conditions, format: [(min1, max1), (min2, max2), ...].
    sigma_rho: Initial C-constraint sigma_rho parameter.
    lambda_reg: InitialRegularizationParameters
    lambda_range: Lambda search range for L-curve method.
    plot: Whether to draw L-curve.
    verbose: Set output verbosity level.
    """

    setup_logging(verbose)

    # Separate known and unknown variable indices
    fixed_indices = sorted(fixed_indices)
    all_indices = set(range(A.shape[1]))
    fixed_indices_set = set(fixed_indices)
    unknown_indices = list(all_indices - fixed_indices_set)
    # logging.debug(f"A columns: {A.shape[1]}, fixed_indices: {fixed_indices}, unknown_indices: {unknown_indices}")
    
    # Partition matrix A
    A_known = A[:, fixed_indices]      # columns for known variables
    A_unknown = A[:, unknown_indices]  # columns for unknown variables
    # logging.debug(f"A_known shape: {A_known.shape}, A_unknown shape: {A_unknown.shape}")
    
    # Modify right-hand side vector
    b_modified = b - A_known @ np.array(fixed_values, dtype=FLOAT)

    sol_to_real_unknown = s2r[unknown_indices,:]
    n = A_unknown.shape[1]
    if x0 is None:
        x0 = np.full(n, initial_rho, dtype=FLOAT)
    initial_rho = np.ones_like(x0) * initial_rho
    weights = np.diag(1.0 / (error**2))
    AtWA = A_unknown.T @ weights @ A_unknown
    # logging.debug(f"A shape: {A_unknown.shape}, rank: {np.linalg.matrix_rank(A_unknown)}")
    # logging.debug(f"AtWA shape: {AtWA.shape}, rank: {np.linalg.matrix_rank(AtWA)}")
    ATWb = A_unknown.T @ weights @ b_modified
    sigma_rho_opt, lambda_rho_opt = None, None

    degrees_of_freedom = A_unknown.shape[0] - A_unknown.shape[1]
    if degrees_of_freedom < 0:
        logging.warning("Warning: Degrees of freedom < 0, possibly due to too many unknown variables or insufficient data.")
        degrees_of_freedom = 0

    if method == 'profile-maximum-likelihood':
        # result0, _ = solve_linear_raw(A_unknown, b_modified, error)
        result0, _ = solve_linear_C(A_unknown, b_modified, error, sol_to_real_unknown, sigma_rho, lambda_rho, x0)
        # sigma_rho_opt, lambda_rho_opt = solve_linear_nelder(A_unknown, b_modified, error, x0, n, sol_to_real_unknown, bounds, sigma_rho, lambda_rho, rho_0=result0)
        sigma_rho_opt, lambda_rho_opt = solve_linear_brute(A_unknown, b_modified, error, x0, sol_to_real_unknown, bounds, result0)
        logging.info(f"Profile maximum likelihood results: sigma_rho : {sigma_rho_opt}, lambda_rho : {lambda_rho_opt}")
        x_solution, x_errors = solve_linear_C(A_unknown, b_modified, error, sol_to_real_unknown, sigma_rho_opt, lambda_rho_opt, x0, params=params)
                                              
    elif method == 'nelder-mead':
        x_solution, sigma_rho_opt, lambda_rho_opt, _ = solve_linear_nelder(A_unknown, b_modified, error, x0, n, sol_to_real_unknown, bounds, sigma_rho, lambda_rho)
        logging.info(f"sigma_rho : {sigma_rho_opt}, lambda_rho : {lambda_rho_opt}")

    elif method == 'C':
        x_solution, x_errors = solve_linear_C(A_unknown, b_modified, error, sol_to_real_unknown, sigma_rho, lambda_rho, x0, params=params)

    elif method == 'l_curve':
        lambda_opt = find_lambda_rho(AtWA, ATWb, A_unknown, b_modified, method, lambda_range=lambda_range, plot=plot)
        x_solution, x_errors = solve_linear_L(A_unknown, b_modified, error, lambda_opt, params=params)
        
    elif method == 'raw':
        x_solution, x_errors = solve_linear_raw(A_unknown, b_modified, error)
    
    elif method == 'Sart':
        # x_solution, x_errors = solve_sart(A_unknown, b_modified, error, x0, relax=1, bounds=bounds, niter=niter, params=params)
        x_solution, x_errors = solve_sart_optimize(A_unknown, b_modified, error, x0, relax=1, niter=niter, params=params)

    elif method == 'L-BFGS-B':
        x_solution, x_errors = solve_scp_minimize(A_unknown, b_modified, error, x0, methods='L-BFGS-B', bounds=bounds, params=params, niter=niter)
    
    elif method == 'trust-constr':
        x_solution, x_errors = solve_scp_minimize(A_unknown, b_modified, error, x0, methods='trust-constr', bounds=bounds, params=params, niter=niter)

    elif method == 'ML-EM':
        x_solution, x_errors = solve_MLEM(A_unknown, b_modified, error, x0, bounds=bounds, niter=niter, params=params)

    elif method == 'M-H':
        MH_bounds = lambda x: construct_difference_weighted(initial_rho, x)
        x_solution, x_errors = solve_MH(A_unknown, b_modified, error, x0, solution_num, T, single_step_size, select_rate, solution_length, fixed_indices, initial_rho, bounds = bounds, MH_bounds=MH_bounds, params=params, solution_cutoff=solution_cutoff, accepted_scale=accepted_scale, s2r = sol_to_real_unknown, r2s = r2s, bias_rate = bias_rate, seem_map = seem_map, Cini = None)

    elif method == 'SA':
        MH_bounds = lambda x: construct_difference_weighted(initial_rho, x)
        x_solution, x_errors, history = solve_SA(A_unknown, b_modified, error, x0, T, T_attrate, T_end, single_step_size, select_rate, fixed_indices, bounds = bounds, MH_bounds=MH_bounds, params=params, s2r = sol_to_real_unknown, r2s = r2s, bias_rate = bias_rate)

    else:
        raise ValueError(f"Unknown solve method: {method}")
    chi_squared, reduced_chi_squared = chisquare_analyze(A_unknown, b_modified, x_solution, degrees_of_freedom, errors=error, params=params)
    # Reconstruct full solution
    full_solution = np.zeros(A.shape[1])
    full_solution[fixed_indices] = fixed_values
    full_solution[unknown_indices] = x_solution

    full_error = np.zeros(A.shape[1])
    full_error[fixed_indices] = 0.
    full_error[unknown_indices] = x_errors

    if params != {}:
        params["Method"] = method
    if method == 'SA':
        return full_solution, x_solution, full_error, x_errors, reduced_chi_squared, history 
    return full_solution, x_solution, full_error, x_errors, reduced_chi_squared

In [ ]:
# bounds = (MIN_DENSITY, MAX_DENSITY)

# L-curve method parameters|
n_lambda = 50  # Number of candidate lambda values

# Define lambda search range (log-spaced)
lambda_min = 1e-30
lambda_max = 1e+10
lambda_candidates = np.logspace(np.log10(lambda_min), np.log10(lambda_max), n_lambda)

scale_factor = 1

bounds = [(0., np.inf)]*(ACTUAL_DIM - MODEL_DIM_INUSE)

previous_chi_square = None
iteration_count = 0
method_ini = 'L-BFGS-B'
method = 'M-H'


# Initial data input
_, x_ini, _, _, _ = solve_linear_L2_straint(LENGTH_MATRIX.toarray(), SIGMA_VECTOR, SIGMA_VECTOR,s2r=SOL_TO_REAL,
    sigma_rho=1, lambda_rho=2, initial_rho=2.6,
    bounds = bounds, niter = 500,
    fixed_indices=MODEL_INDEX, fixed_values=MODEL_DENSITY,
    lambda_reg=0.5, lambda_range=lambda_candidates, method=method_ini, plot=False, verbose=1, params=params)

# M-H method
result, result_independent, full_error, errors, reduced_chi_squared = solve_linear_L2_straint(LENGTH_MATRIX.toarray(), SIGMA_VECTOR, SIGMA_VECTOR,s2r=SOL_TO_REAL, r2s = REAL_TO_SOL,
    sigma_rho=1, lambda_rho=1, initial_rho=2.6,
    bounds = bounds, niter = 500,
    fixed_indices=MODEL_INDEX, fixed_values=MODEL_DENSITY,
    lambda_reg=0.5, lambda_range=lambda_candidates, method=method, plot=False, verbose=1, params=params,
    solution_num = 3000, T = 200., MH_bounds = None, single_step_size = 0.05, select_rate = 0.3, solution_length = 5, solution_cutoff = 3000, accepted_scale = 0.1, bias_rate = 0.2, seem_map = None, x0 = x_ini)

# Status codes for algorithm termination:
# # -1: No progress in last iteration.
# #  0: Max iterations exceeded.
# #  1: First-order optimality < tol.
# #  2: Relative change in cost < tol.
# #  3: Unconstrained solution is optimal (no iterations needed).
# # - raw: No regularization or C constraint
# # - C: C constraint only
# # - l_curve: L-curve method for regularization parameter
# # - l_curveC: L-curve method with C constraint# # - nelder-mead: Nelder-Mead simplex method
# cost: objective value at solution = 0.5 * ||Ax - b||^2
# fun: residual vector Ax - b at solution

In [ ]:
errors_cut = 3

if type(result) is scp.optimize.OptimizeResult:
    Result = result.x
else:
    Result = result
print(Result)
print(f"Resultmax: {np.amax(Result)}")
print(f"Resultmin: {np.amin(Result)}")
print(f"mean: {np.mean(Result):.6f}")

In [ ]:
now = datetime.now()

if type(result) is scp.optimize.OptimizeResult:
    Result = result.x
    file_path = f"output/OptimizeResult_{now.month:02d}-{now.day:02d}.npy"
else:
    Result = result
    Error = full_error
    file_path = f"output/SimBox/SimBox_{method}_{now.month:02d}-{now.day:02d}.npy"
    file_path_e = f"output/SimBox/SimBox_Error_{method}_{now.month:02d}-{now.day:02d}.npy"
rho_t = np.full(N_R,-1,dtype=FLOAT)  # rho_t: density per voxel
error_t = np.full(N_R,-1,dtype=FLOAT)  # error_t: density error per voxel
for i_sol in range(ACTUAL_DIM):  # Store density for multi-detector voxels
    i, j, k = SOL_TO_REAL[i_sol]
    if np.count_nonzero(SEEN_MAP[i, j, k]) >= 1 and i>=0 and j>=0 and k>=0:
        rho_t[i, j, k] = Result[i_sol]
        error_t[i, j, k] = Error[i_sol]
np.save(file_path, arr=rho_t)
np.save(file_path_e, arr=error_t)
print(f"Density saved to {file_path}")
print(f"Error saved to {file_path_e}")

base_name = os.path.splitext(file_path)[0]
txt_file = base_name + ".txt"
processed_params = process_dict(params)

with open(txt_file, 'w') as f:
    json.dump(processed_params, f, 
            indent=4,  # Pretty-print indentation
            ensure_ascii=False)  # Allow Unicode display
    
base_name = os.path.splitext(file_path_e)[0]
txt_file = base_name + ".txt"
processed_params = process_dict(params)

with open(txt_file, 'w') as f:
    json.dump(processed_params, f, 
            indent=4,  # Pretty-print indentation
            ensure_ascii=False)  # Allow Unicode display
